# Legal Contract Analyzer — Llama 3.1 8B on Colab T4 (No Fine-Tuning)

Runs fully locally on Colab's T4 GPU using 4-bit quantisation.

**Setup (one time):**
1. Accept the Llama 3.1 licence at https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct
2. Get a HF token at https://huggingface.co/settings/tokens
3. Paste it below

**Pipeline:**
```
PDF → Text extraction → Chunking → BGE embeddings → FAISS index
  → Retrieve top-k chunks → Llama 3.1 8B (4-bit, local) → Answer
```


In [ ]:
!pip -q install -U transformers bitsandbytes accelerate pypdf faiss-cpu sentence-transformers

In [ ]:
!nvidia-smi

In [ ]:
import os

# --- REQUIRED: paste your HF token (Llama 3.1 is gated) ---
os.environ['HF_TOKEN'] = 'YOUR_HF_TOKEN_HERE'

MODEL_NAME = 'meta-llama/Meta-Llama-3.1-8B-Instruct'

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type='nf4'
)

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=os.environ['HF_TOKEN']
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('Loading model in 4-bit (this takes ~2 min on first run)...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    token=os.environ['HF_TOKEN']
)
model.eval()
print('Model ready.')

In [ ]:
from google.colab import files
uploaded = files.upload()
pdf_path = next(iter(uploaded))
print(f'Loaded: {pdf_path}')

In [ ]:
from pypdf import PdfReader

reader = PdfReader(pdf_path)
full_text = ''
for page in reader.pages:
    full_text += page.extract_text() or ''

print(f'Extracted {len(full_text):,} characters across {len(reader.pages)} pages')

In [ ]:
def chunk_text(text, size=1500, overlap=200):
    text = ' '.join(text.split())
    chunks, i = [], 0
    while i < len(text):
        chunks.append(text[i:i + size])
        i += size - overlap
    return chunks

chunks = chunk_text(full_text)
print(f'Created {len(chunks)} chunks')

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

print('Loading BGE embedding model...')
embedder = SentenceTransformer('BAAI/bge-base-en-v1.5')

print('Embedding chunks...')
embeddings = embedder.encode(
    chunks,
    normalize_embeddings=True,
    show_progress_bar=True
).astype('float32')

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print(f'Index ready — {index.ntotal} vectors')

def retrieve(query, k=5):
    q_emb = embedder.encode([query], normalize_embeddings=True).astype('float32')
    scores, indices = index.search(q_emb, k)
    return [(chunks[i], float(scores[0][rank])) for rank, i in enumerate(indices[0])]

In [ ]:
SYSTEM_PROMPT = """You are a legal contract analyst. Answer questions about contracts accurately.

Rules:
- Answer ONLY from the provided context. Do not use outside knowledge.
- Always cite the specific clause or section your answer is based on.
- If the answer is not in the context, say: "This information is not present in the provided contract sections."
- Distinguish between mandatory obligations ('shall') and optional permissions ('may').
- Be precise and concise."""

def ask(question, k=5, max_new_tokens=256, verbose=False):
    """
    Ask a question about the uploaded contract.

    Args:
        question:       Your question about the contract.
        k:              Number of chunks to retrieve (increase for complex questions).
        max_new_tokens: Max length of the answer.
        verbose:        If True, print the retrieved chunks as well.
    """
    results = retrieve(question, k=k)
    context = '\n\n---\n\n'.join([chunk for chunk, _ in results])

    if verbose:
        print('=== Retrieved context ===')
        for i, (chunk, score) in enumerate(results):
            print(f'[Chunk {i+1}, score={score:.3f}]\n{chunk[:300]}...\n')
        print('=' * 40 + '\n')

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': f'Question: {question}\n\nContract sections:\n{context}'}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,        # ignored when do_sample=False, avoids warning
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    gen_tokens = output[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

print('Ready. Use ask("your question") to analyse the contract.')

In [ ]:
print(ask('Who are the parties to this contract?'))

In [ ]:
print(ask('What are the termination rights and notice periods?'))

In [ ]:
print(ask('What are the payment obligations, amounts, and due dates?'))

In [ ]:
print(ask('What confidentiality obligations exist, and how long do they last?'))

In [ ]:
print(ask('Are there any indemnification or liability clauses? What are the limits?'))

In [ ]:
# Full structured contract summary

SUMMARY_QUESTIONS = [
    'Who are the parties to this contract?',
    'What is the purpose and subject matter of the contract?',
    'What is the contract duration and any renewal terms?',
    'What are the payment obligations, amounts, and due dates?',
    'What are the termination rights and required notice periods?',
    'What confidentiality obligations exist?',
    'What are the governing law and dispute resolution provisions?',
    'Are there any liability caps or indemnification clauses?',
]

print('=' * 60)
print('CONTRACT SUMMARY')
print('=' * 60)
for question in SUMMARY_QUESTIONS:
    print(f'\n{question}')
    print('-' * len(question))
    print(ask(question))
    print()

In [ ]:
# Interactive Q&A — type questions, get answers, type 'quit' to stop

print("Type your question and press Enter. Type 'quit' to stop.\n")
while True:
    q = input('Your question: ').strip()
    if q.lower() in ('quit', 'exit', 'q', ''):
        break
    print('\nAnswer:')
    print(ask(q))
    print()